# 🧠 KPCL AI Analyst: Enterprise-Grade Architecture Guide

This notebook provides a deep-dive into the internal logic, security protocols, and state management of the **KPCL Spare Parts Chatbot (v2.0)**. We have transitioned from a simple conversational agent to a **Stateful LangGraph Analyst** backed by **PostgreSQL**.

## ⚙️ 1. High-Level Architecture

The system follows a **Layered Defense & Multi-Tier Execution** strategy. Every user query passes through several sanity checks and optimization layers before touching the LLM.

### 🏆 The 3-Tier Execution Pipeline

1.  **Deterministic Layer (Performance)**: Instantly answers common queries using hardcoded pandas logic (e.g., Total Revenue, Quantities).
2.  **Template Layer (Precision)**: Uses pre-written complex logic for ranking and time-series comparisons (e.g., Top 5 Models, YoY Growth).
3.  **LLM Layer (Mistral + LangGraph)**: Uses the LLM to write custom Python plans for unique, novel questions. This layer features the **Ralph (Self-Correction) Loop**.

## 🛡️ 2. Security & Guardrails

Enterprise data requires strict protection. The analyst uses a **Layered Defense Strategy** to prevent malicious code injection and unauthorized data access.

In [ ]:
# Example of the Security Guardrails Logic
import re

def intent_safety(q):
    # 1. Block Dangerous Keywords
    dangerous = ["import os", "subprocess", "eval(", "exec(", "shutil"]
    if any(d in q.lower() for d in dangerous):
        return False, "Security Block: System access denied."
    
    # 2. Block Mutation (DROP, DELETE, UPDATE)
    mut_words = ["drop", "delete", "update", "remove", "truncate"]
    if any(re.search(rf'\b{w}\b', q.lower()) for w in mut_words):
        return False, "Security Block: Data mutation is not allowed."
    
    return True, "Safe"

print(intent_safety("Can you drop the table 'spare_parts'?"))

## 🧠 3. Advanced Memory: LangGraph & PostgresSaver

We use **LangGraph** to manage the analyst as a stateful graph. Instead of just storing message history, we store the entire context (filters, results, state) as binary checkpoints in **PostgreSQL**.

### 📊 Database Schema
The `PostgresSaver` automatically manages these tables in the `chatbot_db`:
- `checkpoints`: Snapshots of the AnalystState (messages, filters, intent).
- `checkpoint_blobs`: Binary storage for large data objects.
- `checkpoint_writes`: History of state transitions.

## 🔄 4. The Ralph (Self-Correction) Loop

If the LLM generates Python code that fails (due to a typo or logical error), the Analyst **reads the error**, feeds it back to the LLM, and **retries up to 5 times** automatically. This ensures high reliability without user intervention.

In [ ]:
# Pseudocode for the Ralph Loop
error_feedback = []
for attempt in range(1, 6):
    prompt = build_prompt(q, error_feedback, messages)
    code = llm.invoke(prompt)
    result, err = run_in_sandbox(code)
    
    if not err:
        return result # SUCCESS
    
    error_feedback.append((code, err)) # RETRY with feedback

## 🚀 5. Performance Engineering

To handle 65,000+ rows instantly, the analyst uses **Precomputed Indexes**. This allows the bot to filter data in **0.001s** regardless of the dataset size.

- `DF_BY_YEAR`: Instant access to specific years.
- `DF_BY_REGION`: Instant access to region-specific data.
- `ALL_MODELS`: Instant lookup for model name validation.